# Demo 4 · EmbeddingList：页内多向量检索（NASA 手册）

官方 demo `embedding-list-max-sim`：40 页 NASA 系统工程手册，**每页切成多个 patch 各自嵌入**（多向量），检索时 query 与一页的所有 patch 算 MAX_SIM，取最大相似度作为页面得分。
单向量页嵌入会"平均掉"页内细节；多向量才能回答"哪一页讲了这件事"。
官方用 ColSmol 视觉模型（128 维×每页多 patch）；这里用文本嵌入 API（默认百炼 text-embedding-v4，1024 维）替代，机制完全一样。
数据（CC0/公有领域，PDF 来自仓库）：`https://raw.githubusercontent.com/zc277584121/milvus3-demos/main/demos/embedding-list-max-sim/data/nasa-systems-engineering-handbook-rev2/nasa-systems-engineering-handbook-rev2.pdf`

In [1]:
%pip install -q pymilvus==3.0.2 pypdf requests numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\liuxiao\GenericAgent\temp\milvus3_lab\venv\Scripts\python.exe -m pip install --upgrade pip


嵌入 API：任选一个 OpenAI 兼容服务，填自己的 key（本 notebook 通过环境变量 EMB_KEY 读取）。

In [2]:
import io, json, os, re, requests, numpy as np

BASE = "https://raw.githubusercontent.com/zc277584121/milvus3-demos/main/demos/embedding-list-max-sim/data/nasa-systems-engineering-handbook-rev2/"
EMB_EP = os.environ.get("EMB_EP", "https://dashscope.aliyuncs.com/compatible-mode/v1")  # 换成你的提供商
EMB_KEY = os.environ.get("EMB_KEY") or input("粘贴嵌入 API key: ")
EMB_MODEL = os.environ.get("EMB_MODEL", "text-embedding-v4")

open("nasa_seh.pdf", "wb").write(requests.get(BASE + "nasa-systems-engineering-handbook-rev2.pdf", timeout=180).content)
queries = json.loads(requests.get(BASE + "queries.json", timeout=60).text)["queries"]
print("pdf:", os.path.getsize("nasa_seh.pdf") // 1024, "KB |", len(queries), "queries")

from pypdf import PdfReader
reader = PdfReader("nasa_seh.pdf")
print("PDF 总页数:", len(reader.pages))

pdf: 4025 KB | 8 queries
PDF 总页数: 356


In [3]:
# 40 页子集：pdf 页码 -> printed 页码（来自官方 pages/ 目录）
PDF2PRINTED = {64: 51, 66: 53, 69: 56, 73: 60, 77: 64, 79: 66, 81: 68, 86: 73, 91: 78, 92: 79,
               96: 83, 102: 89, 104: 91, 112: 99, 116: 103, 118: 105, 125: 112, 130: 117,
               138: 125, 146: 133, 164: 151, 170: 157, 174: 161, 176: 163, 178: 165, 181: 168,
               182: 169, 184: 171, 185: 172, 189: 176, 197: 184, 198: 185, 206: 193, 208: 195,
               249: 236, 253: 240, 256: 243, 261: 248, 269: 256, 272: 259}

def page_text(pdf_page_no):
    # 提取 PDF 页文本并整理
    raw = reader.pages[pdf_page_no - 1].extract_text() or ""
    lines = [re.sub(r"\s+", " ", l).strip() for l in raw.splitlines()]
    return " ".join(l for l in lines if l)

pages = []
for pdfno, printed in sorted(PDF2PRINTED.items()):
    pages.append({"page_id": f"nasa-seh-printed-{printed:03d}", "printed_page": printed,
                  "text": page_text(pdfno)})
print("页数:", len(pages), "| 每页平均字符:", sum(len(p["text"]) for p in pages) // len(pages))
print("样例:", pages[1]["page_id"], "|", pages[1]["text"][:120])

页数: 40 | 每页平均字符: 1622
样例: nasa-seh-printed-053 | NASA Systems Engineering Handbook Rev 2 53 4.1 Stakeholder Expectations Definition The Stakeholder Expectations Definiti


每页按段落合并成 ~700 字符的 patch，逐块嵌入（batch=10）。

In [4]:
def split_patches(text, target=700):
    sents = [x.strip() for x in re.split(r"(?<=[.!?])\s+", text) if len(x.strip()) > 20]
    chunks, cur = [], ""
    for x in sents:
        if not cur:
            cur = x
        elif len(cur) + len(x) + 1 <= target:
            cur += " " + x
        else:
            chunks.append(cur)
            cur = x
    if cur:
        chunks.append(cur)
    return chunks

def embed(texts, batch=10):
    vecs = []
    for i in range(0, len(texts), batch):
        r = requests.post(EMB_EP.rstrip("/") + "/embeddings",
                          headers={"Authorization": f"Bearer {EMB_KEY}"},
                          json={"model": EMB_MODEL, "input": texts[i:i + batch], "dimensions": 1024},
                          timeout=60)
        r.raise_for_status()
        vecs += [d["embedding"] for d in r.json()["data"]]
    return np.asarray(vecs, dtype=np.float32)

all_patches, rows = [], []
for pg in pages:
    chunks = split_patches(pg["text"])
    rows.append({"page_id": pg["page_id"], "printed_page": pg["printed_page"],
                 "patches": [{"patch_index": i, "text": c} for i, c in enumerate(chunks)]})
    all_patches += chunks
print("patch 总数:", len(all_patches))

vecs = embed(all_patches)
print("向量:", vecs.shape)
k = 0
for row in rows:
    for p in row["patches"]:
        p["patch_embedding"] = vecs[k].tolist()
        k += 1
print("回填完成")

patch 总数: 122


向量: (122, 1024)
回填完成


## 3. 建集合：patches ARRAY<STRUCT> 多向量 + MAX_SIM_COSINE 索引

In [5]:
from pymilvus import MilvusClient, DataType

client = MilvusClient(uri="http://127.0.0.1:19530")
COLL = "demo4_nasa_pages"
if client.has_collection(COLL):
    client.drop_collection(COLL)

schema = client.create_schema(auto_id=False, enable_dynamic_field=False)
schema.add_field("page_id", DataType.VARCHAR, is_primary=True, max_length=64)
schema.add_field("printed_page", DataType.INT64)
patch_struct = client.create_struct_field_schema()
patch_struct.add_field("patch_index", DataType.INT64)
patch_struct.add_field("text", DataType.VARCHAR, max_length=2048)
patch_struct.add_field("patch_embedding", DataType.FLOAT_VECTOR, dim=1024)
schema.add_field("patches", DataType.ARRAY, element_type=DataType.STRUCT,
                 struct_schema=patch_struct, max_capacity=32)

index_params = client.prepare_index_params()
index_params.add_index(field_name="patches[patch_embedding]", index_type="HNSW",
                       metric_type="MAX_SIM_COSINE", params={"M": 16, "efConstruction": 128})
client.create_collection(COLL, schema=schema, index_params=index_params)
client.insert(COLL, rows)
client.flush(COLL)
client.load_collection(COLL)
print("inserted:", client.get_collection_stats(COLL))

inserted: {'row_count': 40}


## 4. 8 条真实问题，看命中的是不是目标页

官方 queries 每条带标准答案（`target_page_ids`）和一个"混淆页"（`hard_negatives`——内容相近但不是答案的页）。

In [6]:
from pymilvus.client.embedding_list import EmbeddingList

hit, tot = 0, 0
for q in queries:
    qvec = embed([q["text"]])[0]
    ql = EmbeddingList(dim=1024, dtype=np.dtype("float32"))
    ql.add(qvec.tolist())
    res = client.search(COLL, data=[ql], anns_field="patches[patch_embedding]",
                        search_params={"metric_type": "MAX_SIM_COSINE", "params": {"ef": 64}},
                        limit=3, output_fields=["printed_page"])
    top = [h["page_id"] for h in res[0]]
    ok = any(t in q["target_page_ids"] for t in top)
    hit += ok
    tot += 1
    print(q["query_id"], "命中" if ok else "未中", "| top3:", top)
print(f"8 条查询 top3 命中率: {hit}/{tot}")

Q01 命中 | top3: ['nasa-seh-printed-064', 'nasa-seh-printed-053', 'nasa-seh-printed-151']


Q02 命中 | top3: ['nasa-seh-printed-051', 'nasa-seh-printed-089', 'nasa-seh-printed-068']


Q03 命中 | top3: ['nasa-seh-printed-103', 'nasa-seh-printed-105', 'nasa-seh-printed-112']


Q04 命中 | top3: ['nasa-seh-printed-165', 'nasa-seh-printed-161', 'nasa-seh-printed-163']


Q05 命中 | top3: ['nasa-seh-printed-171', 'nasa-seh-printed-172', 'nasa-seh-printed-168']


Q06 命中 | top3: ['nasa-seh-printed-195', 'nasa-seh-printed-083', 'nasa-seh-printed-193']


Q07 命中 | top3: ['nasa-seh-printed-066', 'nasa-seh-printed-060', 'nasa-seh-printed-112']


Q08 命中 | top3: ['nasa-seh-printed-083', 'nasa-seh-printed-236', 'nasa-seh-printed-195']
8 条查询 top3 命中率: 8/8


## 五个坑

1. **insert 后必须 flush 再 search**——多向量字段最容易被缓存绊住，返回空结果的九成原因在这里。
2. **MAX_SIM_COSINE 的分数不是余弦值**：它把 query 的每个向量与页内所有 patch 的相似度做最大池化再按 query 向量数累加，所以可以大于 1（示例里 2.x 正常），只用于排序。
3. **patches 必须声明 max_capacity**，超过容量的数据被截断。
4. 查询侧用 `EmbeddingList` 包装向量，`anns_field` 写 `patches[patch_embedding]`——这和普通 `data=[vec]` 是两种调用形态。
5. 每页 patch 数不均衡时，MAX_SIM 天然偏爱 patch 多的页（更多"中奖机会"），工程上要限制每页 patch 数或做长度归一。

In [7]:
client.drop_collection(COLL)
print("done")

done


跑完这条链，三件事值得看：

1. 命中率——文本嵌入 API + 页内切块，对"这句话在手册哪一页"这类问题，top3 命中率有多高；
2. 命中页的 patch 文本，就是"为什么匹配"的答案（官方 demo 前端把它画成高亮块）；
3. 分数数量级——记住 MAX_SIM 分数只用于排序，不要拿去和余弦阈值混用。